# NB02d — EDA: NYPL Historical Menu Corpus

**Phase 2, Notebook 4 of 5.** Full exploration of all 17 562 NYPL menus (Menu.csv). No model load — pandas and matplotlib only.

The NYPL corpus serves two roles in MenuForge: (1) historical style reference for the art director agent, (2) VLM evaluation ground truth via MenuItem.csv. This notebook characterises the corpus scope, identifies the Italian subset, and surfaces the data-honesty limits that constrain both uses.

| Figure | Description |
|---|---|
| `fig11_nypl_temporal.png` | Menus per decade 1850–2000, Italian subset overlaid |
| `fig12_nypl_complexity.png` | Dish-count distribution: full corpus vs Italian subset |
| `fig13_nypl_occasions.png` | Dining context landscape — top occasion types |
| `fig14_nypl_italian_profile.png` | Italian-tagged menus: year × dish count, place type |

Run cells top-to-bottom. Inspect each output before proceeding.

## 1. Environment setup

No GPU, no model load. Menu.csv takes ~1 s to read.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

NYPL_RAW = ROOT / "data" / "raw" / "nypl"
FIG_DIR  = ROOT / "outputs" / "figures" / "nb02d"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

print(f"NYPL raw : {NYPL_RAW}")
print(f"Figures  : {FIG_DIR}")

## 2. Load Menu.csv and parse dates

17 562 rows. Dates require careful handling: the raw column contains a handful of astronomically bad values (year 1, year 2928) from data-entry errors. We restrict to 1840–2000 for all temporal analysis and flag the remainder as `date_unknown`.

In [ ]:
m = pd.read_csv(NYPL_RAW / "Menu.csv", low_memory=False)
print(f"Rows : {len(m)}")
print(f"Cols : {m.columns.tolist()}")
print()

m["date_parsed"] = pd.to_datetime(m["date"], errors="coerce")
m["year"]        = m["date_parsed"].dt.year
m["decade"]      = (m["year"] // 10 * 10).where(m["year"].between(1840, 2000))

bad_dates = m[m["year"].notna() & ~m["year"].between(1840, 2000)]
null_dates = m[m["year"].isna()]
valid      = m[m["year"].between(1840, 2000)]

print(f"Valid date range (1840–2000) : {len(valid)} rows")
print(f"Null / unparseable dates     : {len(null_dates)} rows")
print(f"Out-of-range dates           : {len(bad_dates)} rows  "
      f"(year range: {bad_dates['year'].min():.0f}–{bad_dates['year'].max():.0f})")
print()
print(f"Actual span  : {int(valid['year'].min())} – {int(valid['year'].max())}")
print(f"dish_count   : median {m['dish_count'].median():.0f}  "
      f"mean {m['dish_count'].mean():.1f}  max {m['dish_count'].max()}")
print(f"page_count   : median {m['page_count'].median():.0f}  "
      f"mean {m['page_count'].mean():.1f}  max {m['page_count'].max()}")

## 3. Define the Italian subset

Italian menus are identified by keyword search across five text fields (`name`, `sponsor`, `event`, `venue`, `place`). This surfaces 179 menus — but the match is deliberately broad and includes three distinct sub-populations that matter for how the data gets used:

1. **Italian cuisine** — actual restaurant or event menus from Italy or Italian-American establishments (the useful style-reference corpus).
2. **Italian-American civic events** — banquets organised by Italian immigrant associations in the US; food is Italian but context is American.
3. **Italian Line shipping company** — menus from the Italian flag-carrier ocean liner fleet. Cuisine is international, not specifically Italian. This is the main contamination source.

Splitting these populations is a data-honesty step: only sub-population 1 is suitable as style reference for the art director agent.

In [ ]:
ITALIAN_TERMS = [
    "ITALIAN", "ITALIA", "NAPOLI", "NAPLES", "VENICE", "VENEZIA",
    "ROME", "ROMA", "MILAN", "MILANO", "FLORENCE", "FIRENZE",
    "SICIL", "TORINO", "GENOA", "GENOVA", "RISTORANTE", "TRATTORIA",
    "OSTERIA", "GIRGENTI",
]

SHIPPING_TERMS = ["ITALIAN LINE", "LLOYD ITALIANO", "NAVIGAZIONE GENERALE"]

def tag_italian(row):
    fields = " ".join(
        str(row[c]) for c in ["name", "sponsor", "event", "venue", "place"]
        if c in row.index and pd.notna(row[c])
    ).upper()
    is_ship = any(t in fields for t in SHIPPING_TERMS)
    is_it   = any(t in fields for t in ITALIAN_TERMS)
    if not is_it:
        return "non-italian"
    if is_ship:
        return "italian-line-ship"
    return "italian"

m["italian_tag"] = m.apply(tag_italian, axis=1)

counts = m["italian_tag"].value_counts()
print("Italian tag distribution:")
for tag, n in counts.items():
    print(f"  {tag:<25} {n:>5}  ({n/len(m)*100:.1f}%)")

it_cuisine = m[m["italian_tag"] == "italian"]
it_ship    = m[m["italian_tag"] == "italian-line-ship"]
it_all     = m[m["italian_tag"] != "non-italian"]

print(f"\nItalian cuisine menus : {len(it_cuisine)}")
print(f"Italian Line ship     : {len(it_ship)}")
print(f"Italian (all tagged)  : {len(it_all)}")
print()
print("Italian cuisine — top places:")
print(it_cuisine["place"].value_counts().head(8).to_string())

## 4. Figure 11 — Menus per decade

The 1900–1910 decade alone contains 7 082 menus — 40% of the corpus. This reflects the NYPL digitisation project's sourcing period, not the historical prevalence of menus. It also means that any style analysis treating the corpus as temporally balanced will be dominated by Edwardian-era design conventions.

In [ ]:
dec_all = (
    m[m["decade"].notna()]
    .groupby("decade")
    .size()
    .reindex(range(1850, 2010, 10), fill_value=0)
)
dec_it = (
    m[(m["italian_tag"] == "italian") & m["decade"].notna()]
    .groupby("decade")
    .size()
    .reindex(range(1850, 2010, 10), fill_value=0)
)
dec_ship = (
    m[(m["italian_tag"] == "italian-line-ship") & m["decade"].notna()]
    .groupby("decade")
    .size()
    .reindex(range(1850, 2010, 10), fill_value=0)
)

fig, ax = plt.subplots(figsize=(11, 4))
x = dec_all.index.astype(int)
w = 7

ax.bar(x, dec_all.values, width=w, color="#9db8d2", label="All NYPL menus", alpha=0.85)
ax.bar(x, dec_it.values,  width=w, color="#2E8B57", label="Italian cuisine", alpha=0.9)
ax.bar(x, dec_ship.values, width=w, color="#c05c3a", label="Italian Line (ship)",
       alpha=0.85, bottom=dec_it.values)

ax.set_xlabel("Decade")
ax.set_ylabel("Number of menus")
ax.set_title("Figure 11 — NYPL corpus: menus per decade (n=17 562)")
ax.set_xticks(x)
ax.set_xticklabels([str(int(v)) for v in x], rotation=45, ha="right")
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))

# Annotate peak
peak_dec = int(dec_all.idxmax())
peak_n   = int(dec_all.max())
ax.annotate(f"Peak: {peak_n:,} ({peak_dec}s)",
            xy=(peak_dec, peak_n), xytext=(peak_dec + 12, peak_n * 0.85),
            arrowprops=dict(arrowstyle="->", color="grey"), fontsize=8)

fig.tight_layout()
out = FIG_DIR / "fig11_nypl_temporal.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
print("Decade totals (top 5):")
for dec, n in dec_all.sort_values(ascending=False).head(5).items():
    print(f"  {int(dec)}s : {n:>5}")

## 5. Figure 12 — Dish-count distribution

Dish count tells us how many line-items the VLM needs to extract per menu in Phase 3. The full corpus has a very long right tail (max 4 053) driven by large hotel banquet booklets. We winsorise at the 99th percentile for plotting so the bulk of the distribution is visible. The Italian cuisine subset has a lower median (27 vs 35 overall) — Italian menus in this period tended toward focused set-course structures rather than the sprawling à-la-carte lists common in American hotel menus.

In [ ]:
cap = int(m["dish_count"].quantile(0.99))
print(f"99th-percentile cap applied at {cap} dishes")
print(f"Rows above cap: {(m['dish_count'] > cap).sum()}")
print()

groups = {
    "All NYPL
(n=17 562)":      m["dish_count"].clip(upper=cap),
    "Italian cuisine
(n=%d)" % len(it_cuisine):
        it_cuisine["dish_count"].clip(upper=cap),
    "Italian Line
(n=%d)" % len(it_ship):
        it_ship["dish_count"].clip(upper=cap),
}
colors = ["#9db8d2", "#2E8B57", "#c05c3a"]

fig, ax = plt.subplots(figsize=(9, 4))
data    = list(groups.values())
labels  = list(groups.keys())

vp = ax.violinplot(data, positions=range(len(data)), widths=0.55,
                   showmedians=True, showextrema=True)
for i, (pc, col) in enumerate(zip(vp["bodies"], colors)):
    pc.set_facecolor(col); pc.set_alpha(0.55)
vp["cmedians"].set_color("black")
vp["cbars"].set_color("black")
vp["cmins"].set_color("black")
vp["cmaxes"].set_color("black")

for i, (vals, col) in enumerate(zip(data, colors)):
    ax.scatter([i] * len(vals), vals, color=col, s=4, alpha=0.25, zorder=3)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel(f"Dishes per menu (capped at {cap})")
ax.set_title("Figure 12 — Menu complexity: dish count distribution")

for i, vals in enumerate(data):
    med = vals.median()
    ax.text(i, med + 2, f"med={med:.0f}", ha="center", va="bottom", fontsize=7.5,
            color="black", fontweight="bold")

fig.tight_layout()
out = FIG_DIR / "fig12_nypl_complexity.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
for label, vals in groups.items():
    tag = label.split("\n")[0]
    print(f"  {tag:<20} median={vals.median():.0f}  mean={vals.mean():.1f}  "
          f"p25={vals.quantile(0.25):.0f}  p75={vals.quantile(0.75):.0f}")

## 6. Figure 13 — Dining context landscape

The `occasion` column classifies the type of event. Most values have trailing semicolons from the original data-entry format — cleaned here for display. The dominance of `DAILY` menus confirms the corpus is weighted toward operational restaurant menus (printed fresh each day) rather than special-event keepsakes, which is good for style-reference purposes: daily menus represent standard design conventions, not one-off commemorative typography.

In [ ]:
occ = (
    m["occasion"]
    .dropna()
    .str.strip()
    .str.rstrip(";")
    .str.strip()
    .str.title()
)
occ_counts = occ.value_counts().head(18)

# Italian cuisine breakdown by occasion
it_occ = (
    it_cuisine["occasion"]
    .dropna()
    .str.strip()
    .str.rstrip(";")
    .str.strip()
    .str.title()
    .value_counts()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Full corpus
axes[0].barh(range(len(occ_counts)), occ_counts.values,
             color="#9db8d2", alpha=0.85, edgecolor="white")
axes[0].set_yticks(range(len(occ_counts)))
axes[0].set_yticklabels(occ_counts.index, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel("Number of menus")
axes[0].set_title("All NYPL menus (top 18 occasions)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{int(v):,}"))

# Italian cuisine
top_it_occ = it_occ.head(12)
axes[1].barh(range(len(top_it_occ)), top_it_occ.values,
             color="#2E8B57", alpha=0.85, edgecolor="white")
axes[1].set_yticks(range(len(top_it_occ)))
axes[1].set_yticklabels(top_it_occ.index, fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel("Number of menus")
axes[1].set_title(f"Italian cuisine subset (n={len(it_cuisine)})")

fig.suptitle("Figure 13 — Dining context: occasion types", fontsize=11)
fig.tight_layout()
out = FIG_DIR / "fig13_nypl_occasions.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
print("Full corpus — top 5 occasions:")
for occ_name, n in occ_counts.head(5).items():
    print(f"  {occ_name:<35} {n:>5}  ({n/len(m)*100:.1f}%)")

## 7. Figure 14 — Italian cuisine corpus profile

Each dot is one Italian-cuisine menu. X = publication year, Y = dish count (winsorised at 200 for legibility), size = page count. Colour distinguishes menus located in Italy from those in the US or at sea.

This is the art-director reference population: 153 menus spanning 1880–1990, concentrated in 1890–1910. The Phase 3 VLM evaluation draws from the 30 images already downloaded (NB02c), which are a subset of this population.

In [ ]:
it_plot = it_cuisine[it_cuisine["year"].between(1840, 2000)].copy()
it_plot["dish_cap"]  = it_plot["dish_count"].clip(upper=200)
it_plot["page_size"] = it_plot["page_count"].clip(upper=10) * 12 + 20

def place_region(place):
    if pd.isna(place):
        return "Unknown"
    p = str(place).upper()
    if any(t in p for t in ["ITALY","ITALIA","ROME","MILAN","NAPLES","VENICE",
                              "FLORENCE","SICIL","TORINO","GENOA","GENOVA","GIRGENTI"]):
        return "Italy"
    if any(t in p for t in ["EN ROUTE","SS ","STEAMER","DAMPFER"]):
        return "At sea"
    return "USA / Other"

it_plot["region"] = it_plot["place"].apply(place_region)

region_colors = {"Italy": "#c05c3a", "USA / Other": "#3A86C8", "At sea": "#2E8B57", "Unknown": "#aaa"}
region_order  = ["Italy", "USA / Other", "At sea", "Unknown"]

fig, ax = plt.subplots(figsize=(11, 5))
for region in region_order:
    sub = it_plot[it_plot["region"] == region]
    if sub.empty:
        continue
    ax.scatter(sub["year"], sub["dish_cap"], s=sub["page_size"],
               color=region_colors[region], alpha=0.65, edgecolors="white",
               linewidths=0.4, label=f"{region} (n={len(sub)})", zorder=3)

ax.set_xlabel("Publication year")
ax.set_ylabel("Dishes per menu (capped at 200)")
ax.set_title(f"Figure 14 — Italian cuisine menus in NYPL: year × complexity "
             f"(n={len(it_plot)}, size ∝ page count)")
ax.legend(fontsize=8, framealpha=0.85)
ax.set_xlim(1875, 2000)

# Mark the 30 downloaded images' date range
ax.axvspan(1899, 1980, alpha=0.07, color="gold", label="Downloaded image date range")

fig.tight_layout()
out = FIG_DIR / "fig14_nypl_italian_profile.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
for region in region_order:
    sub = it_plot[it_plot["region"] == region]
    if sub.empty:
        continue
    print(f"  {region:<15} n={len(sub):>3}  "
          f"dish median={sub['dish_count'].median():.0f}  "
          f"year range {int(sub['year'].min())}–{int(sub['year'].max())}")

## 8. Summary card — NB02d

Key numbers and data-honesty notes for the horizon scan.

In [ ]:
print("=" * 62)
print("NB02d SUMMARY — NYPL historical menu corpus")
print("=" * 62)

print(f"\n── Corpus scope ──")
print(f"  Total menus             : {len(m):>6}")
print(f"  Valid date range        : {int(valid['year'].min())}–{int(valid['year'].max())}")
print(f"  Bad / null dates        : {len(m) - len(valid):>6}")

print(f"\n── Temporal concentration ──")
peak_d = int(dec_all.idxmax())
print(f"  Peak decade             : {peak_d}s  ({int(dec_all.max()):,} menus, "
      f"{dec_all.max()/len(m)*100:.0f}% of corpus)")
pre1920 = dec_all[dec_all.index <= 1910].sum()
print(f"  Pre-1920 share          : {pre1920:,}  ({pre1920/len(m)*100:.0f}%)")

print(f"\n── Menu complexity ──")
p99 = int(m['dish_count'].quantile(0.99))
print(f"  dish_count  median={m['dish_count'].median():.0f}  "
      f"mean={m['dish_count'].mean():.1f}  p99={p99}  max={m['dish_count'].max()}")
print(f"  page_count  median={m['page_count'].median():.0f}  "
      f"mean={m['page_count'].mean():.1f}  max={m['page_count'].max()}")

print(f"\n── Italian subset ──")
print(f"  Italian cuisine         : {len(it_cuisine):>4}  ({len(it_cuisine)/len(m)*100:.1f}%)")
print(f"  Italian Line (ship)     : {len(it_ship):>4}  "
      f"(contamination — remove before style reference use)")
print(f"  Italian cuisine in Italy: "
      f"{(it_plot['region']=='Italy').sum():>4} menus")
print(f"  Date span (cuisine)     : "
      f"{int(it_cuisine['year'].dropna().min())}–{int(it_cuisine['year'].dropna().max())}")
print(f"  dish_count median       : {it_cuisine['dish_count'].median():.0f}  "
      f"(vs {m['dish_count'].median():.0f} full corpus)")

print(f"\n── Data honesty ──")
print("  Corpus is 40% 1900s menus — Edwardian-era design conventions dominate.")
print("  'EN ROUTE' (ocean liner) is the most common place — not restaurants.")
print("  'Italian' keyword match includes Italian shipping company menus.")
print("  153 Italian-cuisine menus span 1880–1990 but concentrate in 1890–1910.")
print("  30 downloaded images (NB02c) are a representative sample of this subset.")
print("  MenuItem.csv (1.33M rows) provides dish-level ground truth for Phase 3.")

print(f"\n── Figures saved ──")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  {f.name}")
print()
print("Next: NB02e_retrieval_comparison.ipynb")